# Vision Agents with smolagents

## Let's install the dependencies and login to our HF account to access the Inference API

In [1]:
import os
from dotenv import load_dotenv
from huggingface_hub import login, whoami

load_dotenv(dotenv_path=".env", override=True)
token = os.environ.get("HF_TOKEN")

# Validate before logging in
try:
    login(token=token, add_to_git_credential=False)
    user = whoami()
    print(f"✅ Logged in as: {user['name']}")
except Exception as e:
    print(f"❌ Login failed: {e}")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✅ Logged in as: zaidalissa


## Providing Images at the Start of the Agent's Execution

In this approach, images are passed to the agent at the start and stored as task_images alongside the task prompt. The agent then processes these images throughout its execution.

Consider the case where Alfred wants to verify the identities of the superheroes attending the party. He already has a dataset of images from previous parties with the names of the guests. Given a new visitor's image, the agent can compare it with the existing dataset and make a decision about letting them in.

In this case, a guest is trying to enter, and Alfred suspects that this visitor might be The Joker impersonating Wonder Woman. Alfred needs to verify their identity to prevent anyone unwanted from entering.

Let’s build the example. First, the images are loaded. In this case, we use images from Wikipedia to keep the example minimal, but imagine the possible use-cases!

In [8]:
from PIL import Image
import requests
from io import BytesIO

image_urls = [
    "https://upload.wikimedia.org/wikipedia/commons/e/e8/The_Joker_at_Wax_Museum_Plus.jpg",
    "https://upload.wikimedia.org/wikipedia/en/9/98/Joker_%28DC_Comics_character%29.jpg"
]

Now that we have the images, the agent will tell us wether the guests is actually a superhero (Wonder Woman) or a villian (The Joker).

### Without Smolagents

In [12]:
import anthropic
import base64

import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=".env", override=True)
client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
content = []
for url in image_urls:
    data = base64.standard_b64encode(requests.get(url, headers=headers).content).decode()
    content.append({"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": data}})

content.append({"type": "text", "text": "Describe the costume and makeup that the comic character in these photos is wearing. Tell me if the guest is The Joker or Wonder Woman."})


response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    messages=[{"role": "user", "content": content}],
)

In [13]:
response

Message(id='msg_015TZmdmhxWgUeegoQoRjUuk', container=None, content=[TextBlock(citations=None, text='I can see these images show someone in a classic Joker costume and makeup. The costume features:\n\n**Makeup:**\n- White face paint/makeup covering the entire face\n- Bright red lipstick extending beyond the natural lip line to create an exaggerated, sinister smile\n- Dark eye makeup\n- Green hair (appears to be slicked back)\n\n**Costume:**\n- Purple suit jacket\n- Orange/yellow shirt underneath\n- Purple bow tie or cravat\n\nThis is definitely The Joker costume, not Wonder Woman. The color scheme of purple, green, and orange/yellow along with the distinctive white face paint and red smile are the classic visual elements associated with the Batman villain known as The Joker.', type='text')], model='claude-sonnet-4-20250514', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_inp

### With Smolagents

In [17]:
import anthropic
import base64
import os
import requests
from dotenv import load_dotenv
from smolagents import CodeAgent, LiteLLMModel

load_dotenv(dotenv_path=".env", override=True)

model = LiteLLMModel(model_id="anthropic/claude-sonnet-4-20250514")

image_urls = [
    "https://upload.wikimedia.org/wikipedia/commons/e/e8/The_Joker_at_Wax_Museum_Plus.jpg",
    "https://upload.wikimedia.org/wikipedia/en/9/98/Joker_%28DC_Comics_character%29.jpg",
]

headers = {"User-Agent": "Mozilla/5.0"}
from PIL import Image
from io import BytesIO

images = []
for url in image_urls:
    response = requests.get(url, headers=headers)
    image = Image.open(BytesIO(response.content)).convert("RGB")
    images.append(image)

agent = CodeAgent(
    tools=[],
    model=model,
    max_steps=20,
    verbosity_level=2,
)

response = agent.run(
    "Describe the costume and makeup that the comic character in these photos is wearing. Tell me if the guest is The Joker or Wonder Woman.",
    images=images,
)

print(response)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Describe the costume and makeup that the comic character in these photos is wearing. Tell me if the guest is    │
│ The Joker or Wonder Woman.                                                                                      │
│                                                                                                                 │
╰─ LiteLLMModel - anthropic/claude-sonnet-4-20250514 ─────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: I need to analyze the images provided to describe the costume and makeup of the comic character shown, and
determine if it's The Joker or Wonder Woman. Looking at the images, I can see clear visual elements that will help 
me identify the character and describe their appearance.                                                           
                                                                                                                   
<code>                                                                                                             
# Analyzing the images provided                                                                                    
# Image 1: Shows a person in costume with distinctive makeup                                                       
# Image 2: Shows what appears to be an artistic representation of a comic character                                
                                                                                                                   
# From the first image, I can observe:                                                                             
# - White face makeup/paint                                                                                        
# - Green hair                                                                                                     
# - Red lipstick extending into an exaggerated smile                                                               
# - Purple suit jacket                                                                                             
# - Orange/yellow shirt or vest                                                                                    
# - Purple bow tie or necktie                                                                                      
                                                                                                                   
# From the second image:                                                                                           
# - Green slicked-back hair                                                                                        
# - White face                                                                                                     
# - Red extended smile                                                                                             
# - Dark suit                                                                                                      
# - This appears to be The Joker from Batman comics                                                                
                                                                                                                   
print("Character identification: The Joker")                                                                       
print("\nCostume description:")                                                                                    
print("- Purple suit jacket")                                                                                      
print("- Orange/yellow shirt or vest underneath")                                                                  
print("- Purple bow tie or necktie")                                                                               
print("- Classic formal attire in the Joker's signature colors")                                                   
                                                                                                                   
print("\nMakeup description:")                                                                                     
print("- White face paint covering the entire face")                                                               
print("- Bright green hair (slicked back)")             

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Analyzing the images provided                                                                                  
  # Image 1: Shows a person in costume with distinctive makeup                                                     
  # Image 2: Shows what appears to be an artistic representation of a comic character                              
                                                                                                                   
  # From the first image, I can observe:                                                                           
  # - White face makeup/paint                                                                                      
  # - Green hair                                                                                                   
  # - Red lipstick extending into an exaggerated smile                                                             
  # - Purple suit jacket                                                                                           
  # - Orange/yellow shirt or vest                                                                                  
  # - Purple bow tie or necktie                                                                                    
                                                                                                                   
  # From the second image:                                                                                         
  # - Green slicked-back hair                                                                                      
  # - White face                                                                                                   
  # - Red extended smile                                                                                           
  # - Dark suit                                                                                                    
  # - This appears to be The Joker from Batman comics                                                              
                                                                                                                   
  print("Character identification: The Joker")                                                                     
  print("\nCostume description:")                                                                                  
  print("- Purple suit jacket")                                                                                    
  print("- Orange/yellow shirt or vest underneath")                                                                
  print("- Purple bow tie or necktie")                                                                             
  print("- Classic formal attire in the Joker's signature colors")                                                 
                                                                                                                   
  print("\nMakeup description:")                                                                                   
  print("- White face paint covering the entire face")                                                             
  print("- Bright green hair (slicked back)")                                                                      
  print("- Exaggerated red lipstick extending beyond natural lip line to create a wide, menacing smile")           
  print("- Dark eye makeup/shadowing")                                                                             
  print("- Pale, ghostly complexion")                                                                              
                                                                                                                   
  character_identity = "The Joker"                       

Execution logs:
Character identification: The Joker

Costume description:
- Purple suit jacket
- Orange/yellow shirt or vest underneath
- Purple bow tie or necktie
- Classic formal attire in the Joker's signature colors

Makeup description:
- White face paint covering the entire face
- Bright green hair (slicked back)
- Exaggerated red lipstick extending beyond natural lip line to create a wide, menacing smile
- Dark eye makeup/shadowing
- Pale, ghostly complexion

Out: The Joker

...

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: I need to fix the syntax error in my final answer by properly escaping the quotes and formatting the      
string correctly.                                                                                                  
                                                                                                                   
<code>                                                                                                             
description = """The character in these photos is The Joker.                                                       
                                                                                                                   
Costume Description:                                                                                               
- Purple suit jacket with classic tailored fit                                                                     
- Bright orange/yellow dress shirt underneath                                                                      
- Purple bow tie or necktie creating a formal but sinister appearance                                              
- The color scheme follows The Joker's iconic purple and orange palette                                            
                                                                                                                   
Makeup Description:                                                                                                
- Complete white face paint covering the entire face and neck                                                      
- Bright green hair that appears slicked back                                                                      
- Exaggerated red lipstick extending far beyond the natural lip line to create The Joker's signature wide, menacing
grin                                                                                                               
- Dark eye makeup and shadowing around the eyes                                                                    
- Pale, ghostly complexion that creates a stark, unsettling appearance                                             
- The makeup captures The Joker's classic comic book appearance with the theatrical, over-the-top styling          
                                                                                                                   
This is definitely The Joker from the Batman universe, not Wonder Woman. The distinctive color scheme (purple,     
green, and white), the maniacal grin, and the formal suit are all hallmarks of this iconic Batman[38;2;230;237;243;

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  description = """The character in these photos is The Joker.                                                     
                                                                                                                   
  Costume Description:                                                                                             
  - Purple suit jacket with classic tailored fit                                                                   
  - Bright orange/yellow dress shirt underneath                                                                    
  - Purple bow tie or necktie creating a formal but sinister appearance                                            
  - The color scheme follows The Joker's iconic purple and orange palette                                          
                                                                                                                   
  Makeup Description:                                                                                              
  - Complete white face paint covering the entire face and neck                                                    
  - Bright green hair that appears slicked back                                                                    
  - Exaggerated red lipstick extending far beyond the natural lip line to create The Joker's signature wide,       
  menacing grin                                                                                                    
  - Dark eye makeup and shadowing around the eyes                                                                  
  - Pale, ghostly complexion that creates a stark, unsettling appearance                                           
  - The makeup captures The Joker's classic comic book appearance with the theatrical, over-the-top styling        
                                                                                                                   
  This is definitely The Joker from the Batman universe, not Wonder Woman. The distinctive color scheme (purple,   
  green, and white), the maniacal grin, and the formal suit are all hallmarks of this iconic Batman villain's      
  appearance."""                                                                                                   
                                                                                                                   
  final_answer(description)                                                                                        
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: The character in these photos is The Joker.

Costume Description:
- Purple suit jacket with classic tailored fit
- Bright orange/yellow dress shirt underneath
- Purple bow tie or necktie creating a formal but sinister appearance
- The color scheme follows The Joker's iconic purple and orange palette

Makeup Description:
- Complete white face paint covering the entire face and neck
- Bright green hair that appears slicked back
- Exaggerated red lipstick extending far beyond the natural lip line to create The Joker's signature wide, menacing
grin
- Dark eye makeup and shadowing around the eyes
- Pale, ghostly complexion that creates a stark, unsettling appearance
- The makeup captures The Joker's classic comic book appearance with the theatrical, over-the-top styling

This is definitely The Joker from the Batman universe, not Wonder Woman. The distinctive color scheme (purple, 
green, and white), the maniacal grin, and the formal suit are all hallmarks of this iconic Batman villain's 
appearance.

[Step 3: Duration 10.34 seconds| Input tokens: 14,455 | Output tokens: 937]

The character in these photos is The Joker.

Costume Description:
- Purple suit jacket with classic tailored fit
- Bright orange/yellow dress shirt underneath
- Purple bow tie or necktie creating a formal but sinister appearance
- The color scheme follows The Joker's iconic purple and orange palette

Makeup Description:
- Complete white face paint covering the entire face and neck
- Bright green hair that appears slicked back
- Exaggerated red lipstick extending far beyond the natural lip line to create The Joker's signature wide, menacing grin
- Dark eye makeup and shadowing around the eyes
- Pale, ghostly complexion that creates a stark, unsettling appearance
- The makeup captures The Joker's classic comic book appearance with the theatrical, over-the-top styling

This is definitely The Joker from the Batman universe, not Wonder Woman. The distinctive color scheme (purple, green, and white), the maniacal grin, and the formal suit are all hallmarks of this iconic Batman villain's appe

## Providing Images with Dynamic Retrieval

This examples is provided as a .py file since it needs to be run locally since it'll browse the web. Go to the Hugging Face Agents Course for more details.